In [1]:
# ============================================================
# CELL 1 — Install unrar & extract dataset
# ============================================================
!apt-get install -y unrar -q
!unrar x "/content/mimic-iv-ext-direct-1.0.rar" "/content/" -y

Reading package lists...
Building dependency tree...
Reading state information...
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal

Unexpected end of archive

Extracting from /content/mimic-iv-ext-direct-1.0.rar

Creating    /content/mimic-iv-ext-direct-1.0                          OK
Creating    /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0  OK
Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/.DS_Store       0%  OK 
Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/app.py       0%  OK 
Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/data_processor.py       0%  OK 
Creating    /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/diagnostic_kg  OK
Creating    /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/diagnostic_kg/

In [2]:
# ============================================================
# CELL 2 — Install all dependencies
# ============================================================
!pip install -q faiss-cpu
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q sentence-transformers
!pip install -q google-genai

print("✅ All packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
✅ All packages installed


In [3]:
# ============================================================
# CELL 3 — Data Extraction from MIMIC-IV
# ============================================================
import os
import json
from typing import List, Dict

BASE_PATH = "/content"
KG_PATH = os.path.join(BASE_PATH, "mimic-iv-ext-direct-1.0", "mimic-iv-ext-direct-1.0.0", "diagnostic_kg", "Diagnosis_flowchart")
CASES_PATH = os.path.join(BASE_PATH, "mimic-iv-ext-direct-1.0", "mimic-iv-ext-direct-1.0.0", "Finished")


def extract_knowledge_chunks() -> List[Dict]:
    """Extract risk factors and symptoms from Knowledge Graph JSON files"""
    chunks = []
    for filename in os.listdir(KG_PATH):
        if not filename.endswith('.json'):
            continue
        with open(os.path.join(KG_PATH, filename), 'r') as f:
            data = json.load(f)
        condition = filename.replace('.json', '')
        for stage_name, stage_data in data.get('knowledge', {}).items():
            if not isinstance(stage_data, dict):
                continue
            if stage_data.get('Risk Factors'):
                chunks.append({
                    'text': f"{condition} - Risk Factors: {stage_data['Risk Factors']}",
                    'source': 'knowledge_graph',
                    'type': 'risk_factors',
                    'condition': condition
                })
            if stage_data.get('Symptoms'):
                chunks.append({
                    'text': f"{condition} - Symptoms: {stage_data['Symptoms']}",
                    'source': 'knowledge_graph',
                    'type': 'symptoms',
                    'condition': condition
                })
    return chunks


def extract_reasoning(data) -> str:
    """Recursively extract reasoning text using $Cause_ pattern"""
    lines = []
    if isinstance(data, dict):
        for key, value in data.items():
            if '$Cause_' in str(key):
                text = str(key).split('$Cause_')[0].strip()
                if text:
                    lines.append(text)
            if isinstance(value, (dict, list)):
                nested = extract_reasoning(value)
                if nested:
                    lines.append(nested)
    elif isinstance(data, list):
        for item in data:
            nested = extract_reasoning(item)
            if nested:
                lines.append(nested)
    return "\n".join(lines)


def extract_case_chunks() -> List[Dict]:
    """Extract patient narratives and reasoning from case JSON files"""
    chunks = []
    for condition_folder in os.listdir(CASES_PATH):
        condition_path = os.path.join(CASES_PATH, condition_folder)
        if not os.path.isdir(condition_path):
            continue
        for root, dirs, files in os.walk(condition_path):
            for filename in files:
                if not filename.endswith('.json'):
                    continue
                with open(os.path.join(root, filename), 'r') as f:
                    data = json.load(f)
                case_id = filename.replace('.json', '')

                # Narrative
                narrative_parts = []
                for i in range(1, 7):
                    val = data.get(f'input{i}', '')
                    if val:
                        narrative_parts.append(f"input{i}: {val}")
                if narrative_parts:
                    chunks.append({
                        'text': f"Case {case_id} [{condition_folder}]\nNarrative:\n" + "\n".join(narrative_parts),
                        'source': 'patient_case',
                        'type': 'narrative',
                        'condition': condition_folder,
                        'case_id': case_id
                    })

                # Reasoning
                for key in data:
                    if not key.startswith('input'):
                        reasoning = extract_reasoning(data[key])
                        if reasoning:
                            chunks.append({
                                'text': f"Case {case_id} [{condition_folder}]\nReasoning:\n{reasoning}",
                                'source': 'patient_case',
                                'type': 'reasoning',
                                'condition': condition_folder,
                                'case_id': case_id
                            })
    return chunks


# Run extraction
print("🚀 Extracting data...")
knowledge_chunks = extract_knowledge_chunks()
case_chunks = extract_case_chunks()
all_chunks = knowledge_chunks + case_chunks

print(f"\n📊 EXTRACTION SUMMARY:")
print(f"  Knowledge chunks : {len(knowledge_chunks)}")
print(f"  Case chunks      : {len(case_chunks)}")
print(f"  Total            : {len(all_chunks)}")

🚀 Extracting data...

📊 EXTRACTION SUMMARY:
  Knowledge chunks : 48
  Case chunks      : 1022
  Total            : 1070


In [4]:
# ============================================================
# CELL 4 — Build FAISS Vector Store with LangChain
# ============================================================
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Convert chunks to LangChain Documents
documents = [
    Document(
        page_content=chunk['text'],
        metadata={
            'source': chunk.get('source', ''),
            'type': chunk.get('type', ''),
            'condition': chunk.get('condition', ''),
        }
    )
    for chunk in all_chunks
]

print(f"📄 Total documents prepared: {len(documents)}")
print("⏳ Loading embedding model (all-MiniLM-L6-v2)...")

# HuggingFace embeddings — runs locally, no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'},   # GPU acceleration on Colab T4
    encode_kwargs={'normalize_embeddings': True}
)

print("⏳ Building FAISS index (this may take a minute)...")
vectorstore = FAISS.from_documents(documents, embeddings)

# Save index to disk (optional but good practice)
vectorstore.save_local("/content/faiss_medical_index")

print("✅ FAISS vector store built and saved!")
print(f"✅ Total vectors indexed: {vectorstore.index.ntotal}")

📄 Total documents prepared: 1070
⏳ Loading embedding model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Building FAISS index (this may take a minute)...
✅ FAISS vector store built and saved!
✅ Total vectors indexed: 1070


In [5]:
# ============================================================
# CELL 5 — Setup Retriever & Test Similarity Search
# ============================================================

# Create retriever — top 5 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Quick test
print("🧪 Testing retriever...\n")
test_query = "What are the symptoms of migraine?"
results = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Top {len(results)} results:\n")
for i, doc in enumerate(results, 1):
    print(f"--- Result {i} [{doc.metadata.get('type', '')}] ---")
    print(doc.page_content[:200])
    print()

🧪 Testing retriever...

Query: What are the symptoms of migraine?
Top 5 results:

--- Result 1 [symptoms] ---
Migraine - Symptoms: headache on one side, pulsating pain, Moderate to severe pain intensity, nausea or vomiting, Sensitivity to light, sound or smell; etc.

--- Result 2 [risk_factors] ---
Migraine - Risk Factors: Genetic predispositions, environmental triggers (such as stress, certain foods or drinks, hormonal changes, changes in sleep patterns; etc.

--- Result 3 [reasoning] ---
Case 17676552-DS-10 [Migraine]
Reasoning:
Blurred vision can be one of the visual symptoms associated with migraines, called a "visual aura." Before or during a migraine attack, some people may experi

--- Result 4 [reasoning] ---
Case 18427803-DS-5 [Migraine]
Reasoning:
Difficulty expressing language may be associated with migraine, especially when migraine is accompanied by neurological symptoms
Persistent headaches and impai

--- Result 5 [reasoning] ---
Case 18805216-DS-21 [Migraine]
Reasoning:
C

In [6]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 13.8 MB/s eta 0:00:00


In [7]:
# ============================================================
# CELL 6 — Medical AI with Groq (FREE - Llama 3.3 70B)
# ============================================================
from groq import Groq

# Free API key: https://console.groq.com  (signup karo, free hai)
GROQ_API_KEY = "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"
groq_client = Groq(api_key=GROQ_API_KEY)


def ask_medical_ai(question: str, top_k: int = 5) -> str:
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks from FAISS
    2. Build prompt with context
    3. Send to Groq (Llama 3.3 70B) - FREE
    4. Return answer
    """
    # Step 1: Retrieve
    retriever_local = vectorstore.as_retriever(search_kwargs={"k": top_k})
    relevant_docs = retriever_local.invoke(question)
    context = "\n\n---\n\n".join([doc.page_content for doc in relevant_docs])

    # Step 2: Build prompt
    prompt = f"""You are an expert medical AI assistant. Answer the question using ONLY the provided medical context.
If the context does not contain enough information, clearly state what is missing.

=== MEDICAL CONTEXT ===
{context}

=== QUESTION ===
{question}

=== ANSWER ===
Provide a clear, structured, and accurate medical answer based on the context above."""

    # Step 3: Generate with Groq
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",  # Free, very powerful
            messages=[
                {"role": "system", "content": "You are an expert medical AI assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=1024,
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ Groq Error: {e}"


print("✅ Medical AI ready!")
print("🤖 Groq Llama 3.3 70B connected (FREE)\n")

✅ Medical AI ready!
🤖 Groq Llama 3.3 70B connected (FREE)



In [8]:
# ============================================================
# CELL 7 — Run Medical Queries
# ============================================================

test_questions = [
    "What are the diagnostic criteria for migraine?",
    "How is chest pain evaluated in emergency settings?",
    "What are the common causes of upper GI bleeding and how are they diagnosed?"
]

print("🤖 MEDICAL AI RESPONSES\n")
print("=" * 80)

for i, question in enumerate(test_questions, 1):
    print(f"\nQ{i}: {question}")
    print("-" * 60)
    answer = ask_medical_ai(question, top_k=5)
    print(f"A{i}: {answer}")
    print("=" * 80)

🤖 MEDICAL AI RESPONSES


Q1: What are the diagnostic criteria for migraine?
------------------------------------------------------------
A1: Based on the provided medical context, the diagnostic criteria for migraine are not explicitly stated. However, the context describes various symptoms and characteristics associated with migraines, including:

1. **Headache characteristics**:
	* Headache on one side
	* Pulsating pain
	* Moderate to severe pain intensity
2. **Associated symptoms**:
	* Nausea or vomiting
	* Sensitivity to light, sound, or smell
	* Blurred vision
	* Eye pain
	* Chills or rigors (may be part of a migraine aura)
3. **Neurological symptoms**:
	* Difficulty expressing language
	* Impaired speech
	* Reduced speech output
	* Abnormal coordination and cerebellar function
	* Impaired balance (positive Romberg test)
4. **Triggers and risk factors**:
	* Genetic predispositions
	* Environmental triggers (stress, certain foods or drinks, hormonal changes, changes in sleep patter

In [9]:
# ============================================================
# CELL 8 — Interactive Chat (ask your own questions)
# ============================================================

while True:
    user_input = input("\n💬 Your medical question (type 'exit' to quit): ").strip()
    if user_input.lower() in ['exit', 'quit', 'q']:
        print("👋 Exiting chat.")
        break
    if not user_input:
        continue
    print("\n🔍 Searching knowledge base...")
    answer = ask_medical_ai(user_input, top_k=5)
    print(f"\n🤖 Answer:\n{answer}")


💬 Your medical question (type 'exit' to quit): causes of heart attacks in cockroaches

🔍 Searching knowledge base...

🤖 Answer:
The provided medical context does not contain any information about cockroaches or their cardiovascular systems. The context only discusses human medical cases related to pulmonary embolism and acute coronary syndrome. 

To answer the question about the causes of heart attacks in cockroaches, we would need information about the cardiovascular system of cockroaches, which is not provided in the given context. Cockroaches are insects and have a different physiology compared to humans, and their "heart" is a tube-like structure that pumps hemolymph, not blood. 

Therefore, based on the provided medical context, it is not possible to provide an accurate answer to the question about the causes of heart attacks in cockroaches. More information about insect physiology and specific studies on cockroach cardiovascular health would be required.

💬 Your medical question

In [10]:
# ============================================================
# MEDICAL ASSISTANT - GRADIO INTERFACE FOR COLAB
# ============================================================

# Step 1: Install dependencies
!pip install -q gradio langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu groq

# Step 2: Install unrar and extract dataset
!apt-get install -y unrar -q

# Extract your RAR file (make sure it's uploaded to Colab)
import os
if os.path.exists("/content/mimic-iv-ext-direct-1.0.rar"):
    !unrar x "/content/mimic-iv-ext-direct-1.0.rar" "/content/" -y
    print("✅ Dataset extracted!")
else:
    print("⚠️ Please upload mimic-iv-ext-direct-1.0.rar to /content/")

# Step 3: Import libraries
import gradio as gr
import json
import os
from typing import List, Dict
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from groq import Groq

# Step 4: Data extraction functions
def extract_knowledge_chunks(kg_path: str) -> List[Dict]:
    """Extract risk factors and symptoms from Knowledge Graph JSON files"""
    chunks = []
    if not os.path.exists(kg_path):
        return chunks

    for filename in os.listdir(kg_path):
        if not filename.endswith('.json'):
            continue
        try:
            with open(os.path.join(kg_path, filename), 'r', encoding='utf-8') as f:
                data = json.load(f)
            condition = filename.replace('.json', '')
            for stage_name, stage_data in data.get('knowledge', {}).items():
                if not isinstance(stage_data, dict):
                    continue
                if stage_data.get('Risk Factors'):
                    chunks.append({
                        'text': f"{condition} - Risk Factors: {stage_data['Risk Factors']}",
                        'source': 'knowledge_graph',
                        'type': 'risk_factors',
                        'condition': condition
                    })
                if stage_data.get('Symptoms'):
                    chunks.append({
                        'text': f"{condition} - Symptoms: {stage_data['Symptoms']}",
                        'source': 'knowledge_graph',
                        'type': 'symptoms',
                        'condition': condition
                    })
        except Exception as e:
            print(f"Error processing {filename}: {e}")
    return chunks


def extract_reasoning(data) -> str:
    """Recursively extract reasoning text using $Cause_ pattern"""
    lines = []
    if isinstance(data, dict):
        for key, value in data.items():
            if '$Cause_' in str(key):
                text = str(key).split('$Cause_')[0].strip()
                if text:
                    lines.append(text)
            if isinstance(value, (dict, list)):
                nested = extract_reasoning(value)
                if nested:
                    lines.append(nested)
    elif isinstance(data, list):
        for item in data:
            nested = extract_reasoning(item)
            if nested:
                lines.append(nested)
    return "\n".join(lines)


def extract_case_chunks(cases_path: str) -> List[Dict]:
    """Extract patient narratives and reasoning from case JSON files"""
    chunks = []
    if not os.path.exists(cases_path):
        return chunks

    for condition_folder in os.listdir(cases_path):
        condition_path = os.path.join(cases_path, condition_folder)
        if not os.path.isdir(condition_path):
            continue
        for root, dirs, files in os.walk(condition_path):
            for filename in files:
                if not filename.endswith('.json'):
                    continue
                try:
                    with open(os.path.join(root, filename), 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    case_id = filename.replace('.json', '')

                    # Narrative
                    narrative_parts = []
                    for i in range(1, 7):
                        val = data.get(f'input{i}', '')
                        if val:
                            narrative_parts.append(f"input{i}: {val}")
                    if narrative_parts:
                        chunks.append({
                            'text': f"Case {case_id} [{condition_folder}]\nNarrative:\n" + "\n".join(narrative_parts),
                            'source': 'patient_case',
                            'type': 'narrative',
                            'condition': condition_folder,
                            'case_id': case_id
                        })

                    # Reasoning
                    for key in data:
                        if not key.startswith('input'):
                            reasoning = extract_reasoning(data[key])
                            if reasoning:
                                chunks.append({
                                    'text': f"Case {case_id} [{condition_folder}]\nReasoning:\n{reasoning}",
                                    'source': 'patient_case',
                                    'type': 'reasoning',
                                    'condition': condition_folder,
                                    'case_id': case_id
                                })
                except Exception as e:
                    print(f"Error processing {filename}: {e}")
    return chunks


# Step 5: Load vectorstore
def load_vectorstore():
    """Load or create vectorstore"""
    BASE_PATH = "/content"
    KG_PATH = os.path.join(BASE_PATH, "mimic-iv-ext-direct-1.0", "mimic-iv-ext-direct-1.0.0", "diagnostic_kg", "Diagnosis_flowchart")
    CASES_PATH = os.path.join(BASE_PATH, "mimic-iv-ext-direct-1.0", "mimic-iv-ext-direct-1.0.0", "Finished")

    # Check if extracted data exists
    if not os.path.exists(KG_PATH):
        print("❌ Dataset not found! Please extract the RAR file first.")
        return None, None, None, None

    print("📚 Loading medical knowledge base...")
    knowledge_chunks = extract_knowledge_chunks(KG_PATH)
    case_chunks = extract_case_chunks(CASES_PATH)
    all_chunks = knowledge_chunks + case_chunks

    if not all_chunks:
        print("❌ No data chunks extracted!")
        return None, None, None, None

    print(f"✅ Extracted {len(knowledge_chunks)} knowledge chunks and {len(case_chunks)} case chunks")

    # Convert to Documents
    documents = [
        Document(
            page_content=chunk['text'],
            metadata={
                'source': chunk.get('source', ''),
                'type': chunk.get('type', ''),
                'condition': chunk.get('condition', ''),
            }
        )
        for chunk in all_chunks
    ]

    # Create embeddings and vectorstore
    print("🔧 Building medical knowledge index (this may take a few minutes)...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

    vectorstore = FAISS.from_documents(documents, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    print("✅ Vectorstore ready!")
    return vectorstore, retriever, knowledge_chunks, case_chunks


# Step 6: Initialize Groq client
def get_groq_client():
    """Initialize Groq client with your API key"""
    # Replace with your actual Groq API key
    GROQ_API_KEY = "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"  # <<< PUT YOUR KEY HERE

    if GROQ_API_KEY and GROQ_API_KEY != "gsk_your_actual_api_key_here":
        return Groq(api_key=GROQ_API_KEY)
    else:
        print("⚠️ Please add your Groq API key!")
        return None


# Step 7: Medical AI function
def ask_medical_ai(question, history):
    """RAG pipeline for medical questions"""

    if retriever is None:
        yield "⚠️ Knowledge base not loaded. Please ensure dataset is extracted."
        return

    if groq_client is None:
        yield "⚠️ API key not configured. Please add your Groq API key."
        return

    # Retrieve relevant documents
    relevant_docs = retriever.invoke(question)
    context = "\n\n---\n\n".join([doc.page_content for doc in relevant_docs])

    # Build prompt
    prompt = f"""You are an expert medical AI assistant. Answer the question using ONLY the provided medical context.
If the context does not contain enough information, clearly state what is missing.

=== MEDICAL CONTEXT ===
{context}

=== QUESTION ===
{question}

=== ANSWER ===
Provide a clear, structured, and accurate medical answer based on the context above."""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are an expert medical AI assistant providing accurate, evidence-based information."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=1024,
            temperature=0.3,
            stream=True
        )

        # Stream the response
        partial_text = ""
        for chunk in response:
            if chunk.choices[0].delta.content:
                partial_text += chunk.choices[0].delta.content
                yield partial_text

    except Exception as e:
        yield f"❌ Error: {str(e)}"


# Step 8: Statistics function
def get_stats():
    """Return system statistics"""
    if knowledge_chunks and case_chunks:
        conditions = list(set([chunk.get('condition', 'Unknown') for chunk in knowledge_chunks]))
        stats = f"""
### 📊 Medical Assistant Statistics

| Metric | Value |
|--------|-------|
| **Clinical Guidelines** | {len(knowledge_chunks)} |
| **Patient Cases** | {len(case_chunks)} |
| **Medical Conditions** | {len(conditions)} |
| **Total Documents** | {len(knowledge_chunks) + len(case_chunks)} |

### 🩺 Conditions Covered
"""
        for condition in sorted(conditions)[:15]:
            stats += f"- {condition}\n"
        if len(conditions) > 15:
            stats += f"- ... and {len(conditions) - 15} more\n"

        return stats
    return "⚠️ No data loaded. Please check your dataset."


# Step 9: Load everything
print("=" * 60)
print("🏥 MEDICAL ASSISTANT - INITIALIZING")
print("=" * 60)

# Load vectorstore
vectorstore, retriever, knowledge_chunks, case_chunks = load_vectorstore()

# Initialize Groq
groq_client = get_groq_client()

if groq_client:
    print("✅ Groq API client ready!")
else:
    print("⚠️ Please add your Groq API key to the code!")

print("=" * 60)
print("✅ Ready to launch Gradio interface!")
print("=" * 60)

# Step 10: Create Gradio interface
def create_interface():
    with gr.Blocks(theme=gr.themes.Soft(
        primary_hue="teal",
        secondary_hue="purple",
        neutral_hue="gray"
    ), title="Medical Assistant - Intelligent Healthcare RAG System") as demo:

        gr.Markdown("""
        # 🏥 Medical Assistant
        ### Intelligent Healthcare Information System | Evidence-Based Medical Answers

        Welcome to the Medical Assistant - an intelligent RAG (Retrieval-Augmented Generation) system
        that provides evidence-based medical information from clinical guidelines and patient cases.

        ---
        """)

        with gr.Row():
            with gr.Column(scale=2):
                chatbot = gr.Chatbot(
                    label="Medical Assistant",
                    height=500,
                    bubble_full_width=False,
                    avatar_images=(None, "🏥")
                )

                with gr.Row():
                    msg = gr.Textbox(
                        label="Your Medical Question",
                        placeholder="e.g., What are the symptoms of migraine?",
                        scale=4
                    )
                    send_btn = gr.Button("Send", variant="primary", scale=1)

                clear_btn = gr.Button("Clear Chat")

                gr.Markdown("""
                ### 📝 Example Questions
                Click on any question to try:
                """)

                examples = gr.Examples(
                    examples=[
                        "What are the symptoms of migraine?",
                        "How is chest pain evaluated in emergency settings?",
                        "What are the common causes of upper GI bleeding?",
                        "What are the risk factors for heart failure?",
                        "How is diabetes diagnosed and managed?",
                        "What are the diagnostic criteria for stroke?"
                    ],
                    inputs=[msg]
                )

            with gr.Column(scale=1):
                gr.Markdown("### 📊 System Status")

                stats_text = gr.Markdown(get_stats())

                gr.Markdown("---")
                gr.Markdown("### 🔬 How It Works")
                gr.Markdown("""
                1. **📥 Knowledge Extraction** - Clinical guidelines and patient cases are extracted from MIMIC-IV data
                2. **🔍 Semantic Search** - Your question is matched to relevant medical documents
                3. **🧠 AI Generation** - Llama 3.3 70B generates evidence-based answers
                """)

                gr.Markdown("---")
                gr.Markdown("### ⚠️ Disclaimer")
                gr.Markdown(
                    "> This system provides informational content only. "
                    "Always consult qualified healthcare professionals for medical advice, diagnosis, or treatment.",
                    elem_classes="disclaimer"
                )

                gr.Markdown("---")
                gr.Markdown("### 💡 Features")
                gr.Markdown("""
                - ✅ Evidence-based responses
                - ✅ Clinical guideline extraction
                - ✅ 1000+ patient cases
                - ✅ 48+ medical conditions
                - ✅ Real-time streaming answers
                """)

        # Chat function
        def respond(message, chat_history):
            if not message:
                yield chat_history
                return

            chat_history.append([message, None])
            yield chat_history

            response = ""
            for response_chunk in ask_medical_ai(message, None):
                response = response_chunk
                chat_history[-1][1] = response
                yield chat_history

        # Event handlers
        send_btn.click(respond, [msg, chatbot], [chatbot]).then(
            lambda: "", None, [msg], queue=False
        )

        msg.submit(respond, [msg, chatbot], [chatbot]).then(
            lambda: "", None, [msg], queue=False
        )

        clear_btn.click(lambda: [], None, [chatbot], queue=False)

        # Refresh stats button
        refresh_btn = gr.Button("🔄 Refresh Stats", size="sm")
        refresh_btn.click(lambda: get_stats(), None, [stats_text])

    return demo


# Step 11: Launch the app
if __name__ == "__main__":
    demo = create_interface()
    demo.launch(share=True, debug=False)

Reading package lists...
Building dependency tree...
Reading state information...
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 100 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/mimic-iv-ext-direct-1.0.rar

Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/.DS_Store       0%  OK 
Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/app.py       0%  OK 
Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/data_processor.py       0%  OK 
Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/diagnostic_kg/Diagnosis_flowchart/Acute Coronary Syndrome.json       0%  OK 
Extracting  /content/mimic-iv-ext-direct-1.0/mimic-iv-ext-direct-1.0.0/diagnostic_kg/Diagnosis_flowchart/Adrenal Insufficiency.json       0%  OK 
Extracting  /content/mimic-iv-ext-di

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vectorstore ready!
✅ Groq API client ready!
✅ Ready to launch Gradio interface!


/tmp/ipykernel_3112/4038100978.py:301: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(
/tmp/ipykernel_3112/4038100978.py:319: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3112/4038100978.py:319: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3112/4038100978.py:319: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://485a585f59f31a9ffd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
